# Cleaning Messy Chicago Traffic Crashes Dataset
Dataset from https://catalog.data.gov/dataset/traffic-crashes-crashes

In [1]:
import importlib
from pathlib import Path

import pandas as pd

import lib

importlib.reload(lib)

PROJECT_DIRECTORY = lib.get_project_dir()
PROJECT_NAME = lib.get_project_name()
log = lib.getLogger(PROJECT_NAME)

# Read dataset

In [2]:
filename = PROJECT_DIRECTORY / 'chicago_traffic_crashes.csv'
df = lib.read_data(filename, sep=',', encoding='ascii')

PortfolioLogger.lib.tools: INFO: Encoding: ascii
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from chicago_traffic_crashes.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10b7d22a0> took 5.061 secs to complete.


## Getting the general info to see what we're working with

In [3]:
log.info(f'DataFrame shape {df.shape}')

PortfolioLogger.chicago_traffic_crashes: INFO: DataFrame shape (1010926, 48)


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010926 entries, 0 to 1010925
Data columns (total 48 columns):
 #   Column                         Non-Null Count    Dtype  
---  ------                         --------------    -----  
 0   CRASH_RECORD_ID                1010926 non-null  object 
 1   CRASH_DATE_EST_I               73391 non-null    object 
 2   CRASH_DATE                     1010926 non-null  object 
 3   POSTED_SPEED_LIMIT             1010926 non-null  int64  
 4   TRAFFIC_CONTROL_DEVICE         1010926 non-null  object 
 5   DEVICE_CONDITION               1010926 non-null  object 
 6   WEATHER_CONDITION              1010926 non-null  object 
 7   LIGHTING_CONDITION             1010926 non-null  object 
 8   FIRST_CRASH_TYPE               1010926 non-null  object 
 9   TRAFFICWAY_TYPE                1010926 non-null  object 
 10  LANE_CNT                       199032 non-null   float64
 11  ALIGNMENT                      1010926 non-null  object 
 12  ROADWAY_SURFAC

In [8]:
df.head()

,CRASH_RECORD_ID,CRASH_DATE_EST_I,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,NaN,01/14/2025 12:25:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,12,3,1,41.997808,-87.655770,POINT (-87.655770494712 41.997807727633)
1,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...,NaN,05/23/2025 09:30:00 AM,30,STOP SIGN/FLASHER,UNKNOWN,UNKNOWN,DAYLIGHT,TURNING,DIVIDED - W/MEDIAN (NOT RAISED),...,0.0,0.0,2.0,0.0,9,6,5,41.946529,-87.688106,POINT (-87.688106391039 41.946529480518)
2,04d91dffc94f677358ca47056921ba5c4224320df27ed4...,Y,04/05/2025 08:00:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,...,0.0,0.0,1.0,0.0,20,7,4,41.899325,-87.715074,POINT (-87.715074373867 41.899324573751)
3,0b5603954d84b7341c7cad4f570ea039e85919f3750ccb...,NaN,05/23/2025 09:15:00 AM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PEDALCYCLIST,NOT DIVIDED,...,1.0,0.0,2.0,0.0,9,6,5,41.902793,-87.699412,POINT (-87.699412181285 41.902792968177)
4,00bce77960c2faa2a8782a8cac1d6e5715802d6c072a08...,NaN,01/14/2025 08:00:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,REAR END,FOUR WAY,...,0.0,0.0,2.0,0.0,8,3,1,41.691207,-87.720555,POINT (-87.720554863466 41.691206664451)


### Checking for null indices and rows

In [5]:
null_columns, null_rows, null_columns = lib.null_info(df)

PortfolioLogger.lib.tools: INFO: Null INFO ********************************************
PortfolioLogger.lib.tools: INFO: 26/48 columns contain null values
PortfolioLogger.lib.tools: INFO: 10274099 total null values
PortfolioLogger.lib.tools: INFO: 0 row(s) contain ALL null values
PortfolioLogger.lib.tools: INFO: 0 column(s) contain ALL null values
PortfolioLogger.lib.tools: INFO: Risk Rate INFO ***************************************
PortfolioLogger.lib.tools: INFO: 
                              error_percent error_count
CRASH_RECORD_ID                       0.00%           0
CRASH_DATE_EST_I                     92.74%      937535
CRASH_DATE                            0.00%           0
POSTED_SPEED_LIMIT                    0.00%           0
TRAFFIC_CONTROL_DEVICE                0.00%           0
DEVICE_CONDITION                      0.00%           0
WEATHER_CONDITION                     0.00%           0
LIGHTING_CONDITION                    0.00%           0
FIRST_CRASH_TYPE        

## No rows or columns that are ALL null. Moving onto cleaning columns.

## Creating a backup

In [6]:
df_cp1 = df.copy()

# Processing NaN Values

In [7]:
df_cp1.columns

Index(['CRASH_RECORD_ID', 'CRASH_DATE_EST_I', 'CRASH_DATE',
       'POSTED_SPEED_LIMIT', 'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION',
       'WEATHER_CONDITION', 'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE',
       'TRAFFICWAY_TYPE', 'LANE_CNT', 'ALIGNMENT', 'ROADWAY_SURFACE_COND',
       'ROAD_DEFECT', 'REPORT_TYPE', 'CRASH_TYPE', 'INTERSECTION_RELATED_I',
       'NOT_RIGHT_OF_WAY_I', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE',
       'PHOTOS_TAKEN_I', 'STATEMENTS_TAKEN_I', 'DOORING_I', 'WORK_ZONE_I',
       'WORK_ZONE_TYPE', 'WORKERS_PRESENT_I', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 

## Checking for NaN and Non-NaN columns

In [8]:
# Create a null and non_null columns list
non_null_columns = lib.notnull(df_cp1).columns

log.info(non_null_columns)
log.info(null_columns)

PortfolioLogger.chicago_traffic_crashes: INFO: Index(['CRASH_RECORD_ID', 'CRASH_DATE', 'POSTED_SPEED_LIMIT',
       'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION',
       'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE',
       'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'CRASH_TYPE',
       'DAMAGE', 'DATE_POLICE_NOTIFIED', 'PRIM_CONTRIBUTORY_CAUSE',
       'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO', 'NUM_UNITS', 'CRASH_HOUR',
       'CRASH_DAY_OF_WEEK', 'CRASH_MONTH'],
      dtype='object')
PortfolioLogger.chicago_traffic_crashes: INFO: Empty DataFrame
Columns: []
Index: []


## Checking for columns whose values are 75% or more NaN

In [9]:
# Get columns that mostly have NaN values. Will need to look at these closer.
high_risk = lib.error_rates(df_cp1, err_rate=.75)

Taking a look at the high risk columns

In [10]:
high_risk

,error_percent,error_count
CRASH_DATE_EST_I,92.74%,937535
LANE_CNT,80.31%,811894
INTERSECTION_RELATED_I,77.02%,778632
NOT_RIGHT_OF_WAY_I,95.51%,965549
PHOTOS_TAKEN_I,98.58%,996562
STATEMENTS_TAKEN_I,97.62%,986880
DOORING_I,99.68%,1007681
WORK_ZONE_I,99.46%,1005426
WORK_ZONE_TYPE,99.58%,1006725
WORKERS_PRESENT_I,99.86%,1009513


## Removing data that is not useable

In [11]:
df_cp1 = df_cp1.drop(high_risk.index, axis=1)

## Filling the rest of the NaN values with filler values

In [12]:
df_cp1, filler_values = lib.fillnull(df_cp1)

In [13]:
df_cp1

,CRASH_RECORD_ID,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,ALIGNMENT,...,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION
0,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,01/14/2025 12:25:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,...,0.0,0.0,2.0,0.0,12,3,1,41.997808,-87.655770,POINT (-87.655770494712 41.997807727633)
1,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...,05/23/2025 09:30:00 AM,30,STOP SIGN/FLASHER,UNKNOWN,UNKNOWN,DAYLIGHT,TURNING,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,...,0.0,0.0,2.0,0.0,9,6,5,41.946529,-87.688106,POINT (-87.688106391039 41.946529480518)
2,04d91dffc94f677358ca47056921ba5c4224320df27ed4...,04/05/2025 08:00:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,STRAIGHT AND LEVEL,...,0.0,0.0,1.0,0.0,20,7,4,41.899325,-87.715074,POINT (-87.715074373867 41.899324573751)
3,0b5603954d84b7341c7cad4f570ea039e85919f3750ccb...,05/23/2025 09:15:00 AM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PEDALCYCLIST,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,2.0,0.0,9,6,5,41.902793,-87.699412,POINT (-87.699412181285 41.902792968177)
4,00bce77960c2faa2a8782a8cac1d6e5715802d6c072a08...,01/14/2025 08:00:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,REAR END,FOUR WAY,STRAIGHT AND LEVEL,...,0.0,0.0,2.0,0.0,8,3,1,41.691207,-87.720555,POINT (-87.720554863466 41.691206664451)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010921,7a7963ac15a38f3cf7549397dd053fd7612dd5194de280...,10/18/2025 03:50:00 AM,35,NO CONTROLS,NO CONTROLS,RAIN,DARKNESS,FIXED OBJECT,NOT DIVIDED,STRAIGHT AND LEVEL,...,0.0,0.0,1.0,0.0,3,7,10,41.899200,-87.618841,POINT (-87.618840672611 41.8992002546)
1010922,84e546c12fd8fd78ae4e2dc26fdb02d6e8f6c4c92be450...,10/18/2025 05:08:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,NOT DIVIDED,STRAIGHT AND LEVEL,...,0.0,0.0,5.0,0.0,17,7,10,41.848081,-87.675881,POINT (-87.675881338024 41.848080595588)
1010923,8237045757e7fed5adfb450bd7c8472e28a61d322f86bb...,10/19/2025 01:50:00 AM,30,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,REAR TO FRONT,NOT DIVIDED,STRAIGHT AND LEVEL,...,0.0,0.0,5.0,0.0,1,1,10,41.949157,-87.648527,POINT (-87.648526617888 41.949157243366)
1010924,61c8dcd63fae60613bc9ec526fa901420cbe99a6d35840...,07/10/2023 12:29:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLEAR,DAYLIGHT,TURNING,FOUR WAY,STRAIGHT AND LEVEL,...,0.0,0.0,2.0,0.0,12,2,7,41.857531,-87.644929,POINT (-87.644928607359 41.857530859236)


## Creating a backup

In [14]:
df_cp2 = df_cp1.copy()

In [15]:
df_cp2.columns

Index(['CRASH_RECORD_ID', 'CRASH_DATE', 'POSTED_SPEED_LIMIT',
       'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION',
       'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE',
       'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE',
       'CRASH_TYPE', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 'LONGITUDE', 'LOCATION'],
      dtype='object')

# Checking and Validating data

In [16]:
df_cp2.loc[:, 'CRASH_DATE']

0          01/14/2025 12:25:00 PM
1          05/23/2025 09:30:00 AM
2          04/05/2025 08:00:00 PM
3          05/23/2025 09:15:00 AM
4          01/14/2025 08:00:00 AM
                    ...          
1010921    10/18/2025 03:50:00 AM
1010922    10/18/2025 05:08:00 PM
1010923    10/19/2025 01:50:00 AM
1010924    07/10/2023 12:29:00 PM
1010925    10/13/2019 01:40:00 AM
Name: CRASH_DATE, Length: 1010926, dtype: object

Unifying proper formatting for pd.Timestamp data type

In [17]:
df_cp2.loc[:, 'POSTED_SPEED_LIMIT']

0          30
1          30
2          30
3          30
4          30
           ..
1010921    35
1010922    30
1010923    30
1010924    30
1010925    30
Name: POSTED_SPEED_LIMIT, Length: 1010926, dtype: int64

Quick check of values

In [18]:
minimum = df_cp2.loc[:, 'POSTED_SPEED_LIMIT'].min()
maximum = df_cp2.loc[:, 'POSTED_SPEED_LIMIT'].max()
average = df_cp2.loc[:, 'POSTED_SPEED_LIMIT'].mean()
log.info(f'Min/Max/Avg POSTED_SPEED_LIMIT {minimum}/{maximum}/{average}')

PortfolioLogger.chicago_traffic_crashes: INFO: Min/Max/Avg POSTED_SPEED_LIMIT 0/99/28.422826200928654


Valid values, no need to clean this column

Checking for categories

In [19]:
df_cp2.loc[:, 'DEVICE_CONDITION'].unique()

array(['FUNCTIONING PROPERLY', 'UNKNOWN', 'NO CONTROLS',
       'FUNCTIONING IMPROPERLY', 'OTHER', 'NOT FUNCTIONING',
       'WORN REFLECTIVE MATERIAL', 'MISSING'], dtype=object)

In [20]:
df_cp2.loc[:, 'WEATHER_CONDITION'].unique()

array(['SNOW', 'UNKNOWN', 'CLEAR', 'RAIN', 'CLOUDY/OVERCAST',
       'FREEZING RAIN/DRIZZLE', 'SLEET/HAIL', 'OTHER', 'BLOWING SNOW',
       'SEVERE CROSS WIND GATE', 'FOG/SMOKE/HAZE',
       'BLOWING SAND, SOIL, DIRT'], dtype=object)

To keep with current formatting, switching 'BLOWING SAND, SOIL, DIRT' to '/'

In [21]:
df_cp2.loc[:, 'WEATHER_CONDITION'] = df_cp2.loc[:, 'WEATHER_CONDITION'].str.replace(', ', '/')
df_cp2.loc[:, 'WEATHER_CONDITION'].unique()

array(['SNOW', 'UNKNOWN', 'CLEAR', 'RAIN', 'CLOUDY/OVERCAST',
       'FREEZING RAIN/DRIZZLE', 'SLEET/HAIL', 'OTHER', 'BLOWING SNOW',
       'SEVERE CROSS WIND GATE', 'FOG/SMOKE/HAZE',
       'BLOWING SAND/SOIL/DIRT'], dtype=object)

In [22]:
df_cp2.loc[:, 'LIGHTING_CONDITION'].unique()

array(['DAYLIGHT', 'UNKNOWN', 'DARKNESS, LIGHTED ROAD', 'DARKNESS',
       'DUSK', 'DAWN'], dtype=object)

In [23]:
df_cp2.loc[:, 'FIRST_CRASH_TYPE'].unique()

array(['SIDESWIPE SAME DIRECTION', 'TURNING', 'PARKED MOTOR VEHICLE',
       'PEDALCYCLIST', 'REAR END', 'ANGLE',
       'SIDESWIPE OPPOSITE DIRECTION', 'FIXED OBJECT', 'PEDESTRIAN',
       'REAR TO SIDE', 'REAR TO FRONT', 'HEAD ON', 'OTHER OBJECT',
       'OTHER NONCOLLISION', 'REAR TO REAR', 'OVERTURNED', 'ANIMAL',
       'TRAIN'], dtype=object)

In [24]:
df_cp2['TRAFFICWAY_TYPE'].unique()

array(['DIVIDED - W/MEDIAN (NOT RAISED)', 'NOT DIVIDED', 'FOUR WAY',
       'DIVIDED - W/MEDIAN BARRIER', 'ONE-WAY', 'PARKING LOT', 'UNKNOWN',
       'T-INTERSECTION', 'OTHER', 'TRAFFIC ROUTE', 'DRIVEWAY',
       'UNKNOWN INTERSECTION TYPE', 'ALLEY', 'Y-INTERSECTION',
       'FIVE POINT, OR MORE', 'L-INTERSECTION', 'RAMP',
       'CENTER TURN LANE', 'NOT REPORTED', 'ROUNDABOUT'], dtype=object)

In [25]:
df_cp2['ALIGNMENT'].unique()

array(['STRAIGHT AND LEVEL', 'STRAIGHT ON GRADE', 'STRAIGHT ON HILLCREST',
       'CURVE, LEVEL', 'CURVE ON GRADE', 'CURVE ON HILLCREST'],
      dtype=object)

In [26]:
df_cp2['ROADWAY_SURFACE_COND'].unique()

array(['SNOW OR SLUSH', 'UNKNOWN', 'DRY', 'WET', 'ICE', 'OTHER',
       'SAND, MUD, DIRT'], dtype=object)

In [27]:
df_cp2['ROAD_DEFECT'].unique()

array(['NO DEFECTS', 'UNKNOWN', 'SHOULDER DEFECT', 'WORN SURFACE',
       'RUT, HOLES', 'OTHER', 'DEBRIS ON ROADWAY'], dtype=object)

In [28]:
df_cp2.columns

Index(['CRASH_RECORD_ID', 'CRASH_DATE', 'POSTED_SPEED_LIMIT',
       'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION',
       'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE',
       'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'REPORT_TYPE',
       'CRASH_TYPE', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'BEAT_OF_OCCURRENCE', 'NUM_UNITS',
       'MOST_SEVERE_INJURY', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'CRASH_HOUR', 'CRASH_DAY_OF_WEEK', 'CRASH_MONTH',
       'LATITUDE', 'LONGITUDE', 'LOCATION'],
      dtype='object')

In [29]:
df_cp2['REPORT_TYPE'].unique()

array(['ON SCENE', 'NOT ON SCENE (DESK REPORT)', 'Not Available',
       'AMENDED'], dtype=object)

In [30]:
df_cp2['CRASH_TYPE'].unique()

array(['NO INJURY / DRIVE AWAY', 'INJURY AND / OR TOW DUE TO CRASH'],
      dtype=object)

In [31]:
df_cp2['HIT_AND_RUN_I'].unique()

array(['Y', 'Not Available', 'N'], dtype=object)

In [32]:
df_cp2['DAMAGE'].unique()

array(['$501 - $1,500', 'OVER $1,500', '$500 OR LESS'], dtype=object)

In [33]:
df_cp2['DATE_POLICE_NOTIFIED'].unique()

array(['01/14/2025 12:38:00 PM', '05/27/2025 10:40:00 AM',
       '04/05/2025 09:23:00 PM', ..., '10/19/2025 02:12:00 AM',
       '07/10/2023 01:05:00 PM', '10/13/2019 02:50:00 AM'],
      shape=(765395,), dtype=object)

In [34]:
df_cp2['PRIM_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER TURNING/NO SIGNAL', 'IMPROPER OVERTAKING/PASSING',
       'UNABLE TO DETERMINE', 'FOLLOWING TOO CLOSELY',
       'DISREGARDING TRAFFIC SIGNALS', 'IMPROPER LANE USAGE',
       'FAILING TO YIELD RIGHT-OF-WAY', 'NOT APPLICABLE',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'DISREGARDING STOP SIGN', 'IMPROPER BACKING',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'TURNING RIGHT ON RED', 'PHYSICAL CONDITION OF DRIVER',
       'EQUIPMENT - VEHICLE CONDITION', 'DRIVING ON WRONG SIDE/WRONG WAY',
       'DISREGARDING ROAD MARKINGS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'DISTRACTION - OTHER ELECTRON

In [35]:
df_cp2['SEC_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER OVERTAKING/PASSING', 'UNABLE TO DETERMINE',
       'FOLLOWING TOO CLOSELY', 'NOT APPLICABLE',
       'FAILING TO YIELD RIGHT-OF-WAY',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'PHYSICAL CONDITION OF DRIVER', 'IMPROPER LANE USAGE',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'IMPROPER BACKING', 'DISREGARDING STOP SIGN',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'IMPROPER TURNING/NO SIGNAL', 'ROAD CONSTRUCTION/MAINTENANCE',
       'DRIVING ON WRONG SIDE/WRONG WAY', 'DISREGARDING TRAFFIC SIGNALS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'EQUIPMENT - VEHICLE CONDITION',
       'CELL PHONE USE OTHER THAN TEXTING',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS', 'TEXTING',
  

In [36]:
df_cp2['STREET_NO']

0          6352
1          3555
2          1005
3          2901
4          4001
           ... 
1010921     898
1010922    2416
1010923    3700
1010924    1800
1010925    2357
Name: STREET_NO, Length: 1010926, dtype: int64

In [37]:
df_cp2['STREET_DIRECTION'].unique()

array(['N', 'W', 'S', 'E', 'Not Available'], dtype=object)

In [38]:
df_cp2['STREET_DIRECTION'].value_counts()['Not Available']

np.int64(4)

In [39]:
df_cp2['BEAT_OF_OCCURRENCE']

0          2433.0
1          1921.0
2          1121.0
3          1211.0
4          2211.0
            ...  
1010921    1833.0
1010922    1034.0
1010923    1925.0
1010924    1235.0
1010925    1414.0
Name: BEAT_OF_OCCURRENCE, Length: 1010926, dtype: float64

In [40]:
df_cp2['NUM_UNITS'].unique()

array([ 2,  1,  5,  3,  4,  6,  7, 10,  8, 14, 12, 18, 11,  9, 16, 13, 15])

In [41]:
df_cp2['MOST_SEVERE_INJURY'].unique()

array(['NO INDICATION OF INJURY', 'NONINCAPACITATING INJURY',
       'REPORTED, NOT EVIDENT', 'INCAPACITATING INJURY', 'FATAL',
       'Not Available'], dtype=object)

In [42]:
df_cp2['INJURIES_TOTAL'].unique()

array([ 0.        ,  1.        ,  2.        ,  3.        ,  5.        ,
        6.        ,  4.        ,  0.19808058,  7.        , 11.        ,
        8.        ,  9.        , 19.        , 15.        , 10.        ,
       21.        , 17.        , 12.        , 14.        , 13.        ,
       16.        ])

In [43]:
df_cp2['INJURIES_FATAL'].unique()

array([0.0000000e+00, 1.0000000e+00, 1.1499483e-03, 2.0000000e+00,
       3.0000000e+00, 4.0000000e+00])

In [44]:
df_cp2['INJURIES_INCAPACITATING'].unique()

array([ 0.        ,  1.        ,  2.        ,  0.01928443,  4.        ,
        3.        ,  5.        ,  6.        ,  7.        , 10.        ,
        8.        ])

In [45]:
sorted(df_cp2['INJURIES_NON_INCAPACITATING'].astype(int).unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(18),
 np.int64(19),
 np.int64(21)]

In [46]:
sorted(df_cp2['INJURIES_REPORTED_NOT_EVIDENT'].astype(int).unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(15),
 np.int64(19)]

In [47]:
sorted(df_cp2['INJURIES_NO_INDICATION'].astype(int).unique())

[np.int64(0),
 np.int64(1),
 np.int64(2),
 np.int64(3),
 np.int64(4),
 np.int64(5),
 np.int64(6),
 np.int64(7),
 np.int64(8),
 np.int64(9),
 np.int64(10),
 np.int64(11),
 np.int64(12),
 np.int64(13),
 np.int64(14),
 np.int64(15),
 np.int64(16),
 np.int64(17),
 np.int64(18),
 np.int64(19),
 np.int64(20),
 np.int64(21),
 np.int64(22),
 np.int64(23),
 np.int64(24),
 np.int64(25),
 np.int64(26),
 np.int64(27),
 np.int64(28),
 np.int64(29),
 np.int64(30),
 np.int64(31),
 np.int64(32),
 np.int64(33),
 np.int64(34),
 np.int64(35),
 np.int64(36),
 np.int64(37),
 np.int64(38),
 np.int64(39),
 np.int64(40),
 np.int64(41),
 np.int64(42),
 np.int64(43),
 np.int64(45),
 np.int64(46),
 np.int64(48),
 np.int64(49),
 np.int64(50),
 np.int64(61)]

In [48]:
sorted(df_cp2['INJURIES_UNKNOWN'].astype(int).unique())

[np.int64(0)]

In [49]:
df_cp2['CRASH_HOUR'].unique() > 24

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False])

In [50]:
df_cp2['CRASH_DAY_OF_WEEK'].unique() > 7

array([False, False, False, False, False, False, False])

In [51]:
df_cp2['CRASH_MONTH'].unique() > 12

array([False, False, False, False, False, False, False, False, False,
       False, False, False])

In [52]:
df_cp2['LATITUDE']

0          41.997808
1          41.946529
2          41.899325
3          41.902793
4          41.691207
             ...    
1010921    41.899200
1010922    41.848081
1010923    41.949157
1010924    41.857531
1010925    41.924150
Name: LATITUDE, Length: 1010926, dtype: float64

In [53]:
df_cp2['LONGITUDE']

0         -87.655770
1         -87.688106
2         -87.715074
3         -87.699412
4         -87.720555
             ...    
1010921   -87.618841
1010922   -87.675881
1010923   -87.648527
1010924   -87.644929
1010925   -87.699151
Name: LONGITUDE, Length: 1010926, dtype: float64

In [54]:
df_cp2['LOCATION']

0          POINT (-87.655770494712 41.997807727633)
1          POINT (-87.688106391039 41.946529480518)
2          POINT (-87.715074373867 41.899324573751)
3          POINT (-87.699412181285 41.902792968177)
4          POINT (-87.720554863466 41.691206664451)
                             ...                   
1010921      POINT (-87.618840672611 41.8992002546)
1010922    POINT (-87.675881338024 41.848080595588)
1010923    POINT (-87.648526617888 41.949157243366)
1010924    POINT (-87.644928607359 41.857530859236)
1010925    POINT (-87.699150882692 41.924150305628)
Name: LOCATION, Length: 1010926, dtype: object

## Creating lists to process

In [55]:
to_drop = [
    'BEAT_OF_OCCURRENCE',
    'MOST_SEVERE_INJURY', # Repeated in other INJURY_* columns
    'CRASH_HOUR',  # repeated in CRASH_DATE
    'CRASH_DAY_OF_WEEK',  # repeated in CRASH_DATE
    'CRASH_MONTH',  # repeated in CRASH_DATE
    'NUM_UNITS', # Would need clarification what this means
    'LOCATION'
]

# Convert to Timestamp
to_datetime = [
    'CRASH_DATE',
    'DATE_POLICE_NOTIFIED',
]

to_consolidate = [
    'WEATHER_CONDITION',
    'TRAFFIC_CONTROL_DEVICE',
    'DEVICE_CONDITION',
    'FIRST_CRASH_TYPE', #?
    'ALIGNMENT,'
    'TRAFFICWAY_TYPE',
    'ROADWAY_SURFACE_COND',
    'ROAD_DEFECT',
    'PRIM_CONTRIBUTORY_CAUSE',
    'SEC_CONTRIBUTORY_CAUSE',
    'ROADWAY_SURFACE_COND',
    'LIGHTING_CONDITION',
    'STREET_NAME', # Add this and STREET_NO together
    'STREET_NO'

]

to_bools_manual = [
    'REPORT_TYPE', # Change to ON_SCENE
    'CRASH_TYPE', # Change to CRASH_SEVERE
    'HIT_AND_RUN_I'
]

to_ints_manual = [
    'DAMAGE'
]

to_ints = [
    'INJURIES_TOTAL',
    'INJURIES_FATAL',
    'INJURIES_INCAPACITATING',
    'INJURIES_NON_INCAPACITATING',
    'INJURIES_REPORTED_NOT_EVIDENT',
    'INJURIES_NO_INDICATION'
]

# No need to check these
valid = [
    'POSTED_SPEED_LIMIT',
    'STREET_DIRECTION',
    'LATITUDE',
    'LONGITUDE',
]

keys2 = to_consolidate + to_bools_manual + to_ints_manual + to_ints + to_datetime + to_drop + valid
diff = set(df_cp2.columns).difference(set(keys2))
log.debug(f'Diff {diff}: {df_cp2.shape[1]}/{len(keys2)}')

## Creating a backup

In [56]:
df_cp3 = df_cp2.copy()

## Dropping unneeded Columns

In [57]:
df_cp3 = df_cp3.drop(to_drop, axis=1)

# Begin Cleaning Process

## Stripping whitespaces from DataFrame

In [58]:
df_cp3 = lib.str_strip(df_cp3)

## Converting Datetime columns

In [59]:
for column in to_datetime:
    df_cp3[column] = pd.to_datetime(df_cp3.loc[:, column], format='%m/%d/%Y %I:%M:%S %p')

In [60]:
df_cp3[to_datetime]

,CRASH_DATE,DATE_POLICE_NOTIFIED
0,2025-01-14 12:25:00,2025-01-14 12:38:00
1,2025-05-23 09:30:00,2025-05-27 10:40:00
2,2025-04-05 20:00:00,2025-04-05 21:23:00
3,2025-05-23 09:15:00,2025-05-23 09:16:00
4,2025-01-14 08:00:00,2025-01-14 08:45:00
...,...,...
1010921,2025-10-18 03:50:00,2025-10-19 23:30:00
1010922,2025-10-18 17:08:00,2025-10-19 09:00:00
1010923,2025-10-19 01:50:00,2025-10-19 02:12:00
1010924,2023-07-10 12:29:00,2023-07-10 13:05:00


## Converting Ints columns

In [61]:
df_cp3['DAMAGE']

0          $501 - $1,500
1            OVER $1,500
2            OVER $1,500
3           $500 OR LESS
4            OVER $1,500
               ...      
1010921      OVER $1,500
1010922      OVER $1,500
1010923      OVER $1,500
1010924      OVER $1,500
1010925     $500 OR LESS
Name: DAMAGE, Length: 1010926, dtype: object

In [62]:
# Converting to ints
df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('$500 OR LESS', '-$500')
df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('$501 - $1,500', '-$1500')
df_cp3['DAMAGE'] = df_cp3['DAMAGE'].replace('OVER $1,500', '+$1500')

In [63]:
for column in to_ints:
    df_cp3[column] = df_cp3[column].astype(int)

## Converting Bools columns
-1 for Not Available
0 for False
1 for True

Renaming 'REPORT_TYPE' to 'ON_SCENE'

In [64]:
df_cp3['REPORT_TYPE'].unique()

array(['ON SCENE', 'NOT ON SCENE (DESK REPORT)', 'Not Available',
       'AMENDED'], dtype=object)

Change to ON_SCENE

In [65]:
df_cp3.loc[:, 'REPORT_TYPE'] = df_cp3['REPORT_TYPE'].replace(
    {
        'ON SCENE': 1,
        'NOT ON SCENE (DESK REPORT)': 0,
        'Not Available': -1,
        'AMENDED': 0,
     }
)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_38598/1565432387.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp3.loc[:, 'REPORT_TYPE'] = df_cp3['REPORT_TYPE'].replace(


In [66]:
df_cp3 = df_cp3.rename({'REPORT_TYPE': 'ON_SCENE'}, axis=1)
df_cp3.loc[:, 'ON_SCENE']


0           1
1           0
2           1
3           1
4           0
           ..
1010921     0
1010922     0
1010923     0
1010924    -1
1010925    -1
Name: ON_SCENE, Length: 1010926, dtype: object

In [67]:
df_cp3.columns

Index(['CRASH_RECORD_ID', 'CRASH_DATE', 'POSTED_SPEED_LIMIT',
       'TRAFFIC_CONTROL_DEVICE', 'DEVICE_CONDITION', 'WEATHER_CONDITION',
       'LIGHTING_CONDITION', 'FIRST_CRASH_TYPE', 'TRAFFICWAY_TYPE',
       'ALIGNMENT', 'ROADWAY_SURFACE_COND', 'ROAD_DEFECT', 'ON_SCENE',
       'CRASH_TYPE', 'HIT_AND_RUN_I', 'DAMAGE', 'DATE_POLICE_NOTIFIED',
       'PRIM_CONTRIBUTORY_CAUSE', 'SEC_CONTRIBUTORY_CAUSE', 'STREET_NO',
       'STREET_DIRECTION', 'STREET_NAME', 'INJURIES_TOTAL', 'INJURIES_FATAL',
       'INJURIES_INCAPACITATING', 'INJURIES_NON_INCAPACITATING',
       'INJURIES_REPORTED_NOT_EVIDENT', 'INJURIES_NO_INDICATION',
       'INJURIES_UNKNOWN', 'LATITUDE', 'LONGITUDE'],
      dtype='object')

Renaming to CRASH_SEVERE

In [68]:
df_cp3['CRASH_TYPE'].unique()

array(['NO INJURY / DRIVE AWAY', 'INJURY AND / OR TOW DUE TO CRASH'],
      dtype=object)

In [69]:
df_cp3['CRASH_TYPE'] = df_cp3['CRASH_TYPE'].replace(
    {
        'NO INJURY / DRIVE AWAY': False,
        'INJURY AND / OR TOW DUE TO CRASH': True,
     }
)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_38598/4060254133.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp3['CRASH_TYPE'] = df_cp3['CRASH_TYPE'].replace(


Renaming to HIT_AND_RUN

In [70]:
df_cp3 = df_cp3.rename({'CRASH_TYPE': 'CRASH_SEVERE'}, axis=1)
df_cp3.loc[:, 'CRASH_SEVERE']

0          False
1          False
2          False
3           True
4          False
           ...  
1010921    False
1010922    False
1010923    False
1010924    False
1010925    False
Name: CRASH_SEVERE, Length: 1010926, dtype: bool

In [71]:
df_cp3['HIT_AND_RUN_I'].unique()

array(['Y', 'Not Available', 'N'], dtype=object)

In [72]:
df_cp3['HIT_AND_RUN_I'] = df_cp3['HIT_AND_RUN_I'].replace(
    {
        'Not Available': False,
        'N': False,
        'Y': True,
     }
)
df_cp3 = df_cp3.rename({'HIT_AND_RUN_I': 'HIT_AND_RUN'}, axis=1)
df_cp3.loc[:, 'HIT_AND_RUN']

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_38598/1472556803.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp3['HIT_AND_RUN_I'] = df_cp3['HIT_AND_RUN_I'].replace(


0           True
1           True
2           True
3          False
4          False
           ...  
1010921    False
1010922    False
1010923     True
1010924    False
1010925    False
Name: HIT_AND_RUN, Length: 1010926, dtype: bool

## Creating a backup

In [73]:
df_cp4 = df_cp3.copy()

## Converting Consolidate column
A manual process

In [74]:
df_cp4['WEATHER_CONDITION'].unique()

array(['SNOW', 'UNKNOWN', 'CLEAR', 'RAIN', 'CLOUDY/OVERCAST',
       'FREEZING RAIN/DRIZZLE', 'SLEET/HAIL', 'OTHER', 'BLOWING SNOW',
       'SEVERE CROSS WIND GATE', 'FOG/SMOKE/HAZE',
       'BLOWING SAND/SOIL/DIRT'], dtype=object)

In [75]:
df_cp4['WEATHER_CONDITION'] = df_cp4['WEATHER_CONDITION'].replace({
    'CLEAR': 'Clear',
    'RAIN': 'Rain',
    'FREEZING RAIN/DRIZZLE': 'Rain',
    'SNOW': 'Snow',
    'BLOWING SNOW': 'Snow',
    'SEVERE CROSS WIND GATE': 'Windy',
    'BLOWING SAND/SOIL/DIRT': 'Windy',
    'CLOUDY/OVERCAST': 'Cloudy',
    'FOG/SMOKE/HAZE': 'Fog/Smoke',
    'SLEET/HAIL': 'Hail',

    'OTHER': 'Unknown',
    'UNKNOWN': 'Unknown',
 })

In [76]:
df_cp4['TRAFFIC_CONTROL_DEVICE'].unique()

array(['TRAFFIC SIGNAL', 'STOP SIGN/FLASHER', 'NO CONTROLS', 'UNKNOWN',
       'OTHER', 'LANE USE MARKING', 'PEDESTRIAN CROSSING SIGN',
       'OTHER REG. SIGN', 'OTHER WARNING SIGN', 'YIELD', 'SCHOOL ZONE',
       'FLASHING CONTROL SIGNAL', 'DELINEATORS', 'NO PASSING',
       'RAILROAD CROSSING GATE', 'POLICE/FLAGMAN', 'RR CROSSING SIGN',
       'BICYCLE CROSSING SIGN', 'OTHER RAILROAD CROSSING'], dtype=object)

In [77]:
df_cp4['TRAFFIC_CONTROL_DEVICE'] = df_cp4['TRAFFIC_CONTROL_DEVICE'].replace({
    'TRAFFIC SIGNAL': 'Traffic Signal',
    'STOP SIGN/FLASHER': 'Stop Sign',
    'FLASHING CONTROL SIGNAL': 'Flashing Signal',

    'YIELD': 'Yield',

    'LANE USE MARKING': 'Lane Marking',
    'OTHER REG. SIGN': 'Regulatory Sign',
    'OTHER WARNING SIGN': 'Warning Sign',

    'BICYCLE CROSSING SIGN': 'Crossing Sign - Bicycle',
    'PEDESTRIAN CROSSING SIGN': 'Crossing Sign - Pedestrian',

    'DELINEATORS': 'Delineators',
    'NO PASSING': 'No Passing Sign',

    'SCHOOL ZONE': 'School Zone',

    'POLICE/FLAGMAN': 'Person',

    'RR CROSSING SIGN': 'Railroad Crossing Sign',
    'RAILROAD CROSSING GATE': 'Railroad Crossing Sign',
    'OTHER RAILROAD CROSSING': 'Railroad Crossing Sign',

    'NO CONTROLS': 'No Controls',

    'OTHER': 'Unknown',
    'UNKNOWN': 'Unknown',
})

In [78]:
df_cp4['DEVICE_CONDITION'].unique()

array(['FUNCTIONING PROPERLY', 'UNKNOWN', 'NO CONTROLS',
       'FUNCTIONING IMPROPERLY', 'OTHER', 'NOT FUNCTIONING',
       'WORN REFLECTIVE MATERIAL', 'MISSING'], dtype=object)

In [79]:
df_cp4['DEVICE_CONDITION'] = df_cp4['DEVICE_CONDITION'].replace(
    {
        'FUNCTIONING PROPERLY': True,
        'UNKNOWN': False,
        'NO CONTROLS': False,
        'FUNCTIONING IMPROPERLY': False,
        'OTHER': False,
        'NOT FUNCTIONING': False,
        'WORN REFLECTIVE MATERIAL': False,
        'MISSING': False,
    }
)
df_cp4 = df_cp4.rename({'DEVICE_CONDITION': 'TRAFFIC_DEVICE_FUNCTIONING'}, axis=1)
df_cp4.loc[:, 'TRAFFIC_DEVICE_FUNCTIONING']

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_38598/1568383276.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp4['DEVICE_CONDITION'] = df_cp4['DEVICE_CONDITION'].replace(


0           True
1          False
2          False
3          False
4           True
           ...  
1010921    False
1010922    False
1010923    False
1010924     True
1010925    False
Name: TRAFFIC_DEVICE_FUNCTIONING, Length: 1010926, dtype: bool

In [80]:
df_cp4['FIRST_CRASH_TYPE'].unique()

array(['SIDESWIPE SAME DIRECTION', 'TURNING', 'PARKED MOTOR VEHICLE',
       'PEDALCYCLIST', 'REAR END', 'ANGLE',
       'SIDESWIPE OPPOSITE DIRECTION', 'FIXED OBJECT', 'PEDESTRIAN',
       'REAR TO SIDE', 'REAR TO FRONT', 'HEAD ON', 'OTHER OBJECT',
       'OTHER NONCOLLISION', 'REAR TO REAR', 'OVERTURNED', 'ANIMAL',
       'TRAIN'], dtype=object)

In [81]:
df_cp4['TRAFFICWAY_TYPE'].unique()

array(['DIVIDED - W/MEDIAN (NOT RAISED)', 'NOT DIVIDED', 'FOUR WAY',
       'DIVIDED - W/MEDIAN BARRIER', 'ONE-WAY', 'PARKING LOT', 'UNKNOWN',
       'T-INTERSECTION', 'OTHER', 'TRAFFIC ROUTE', 'DRIVEWAY',
       'UNKNOWN INTERSECTION TYPE', 'ALLEY', 'Y-INTERSECTION',
       'FIVE POINT, OR MORE', 'L-INTERSECTION', 'RAMP',
       'CENTER TURN LANE', 'NOT REPORTED', 'ROUNDABOUT'], dtype=object)

In [82]:
df_cp4['TRAFFICWAY_TYPE'] = df_cp4['TRAFFICWAY_TYPE'].replace({
    'DIVIDED - W/MEDIAN (NOT RAISED)': 'Divided',
    'DIVIDED - W/MEDIAN BARRIER': 'Divided',
    'NOT DIVIDED': 'Not Divided',

    'T-INTERSECTION': 'Intersection',
    'Y-INTERSECTION': 'Intersection',
    'L-INTERSECTION': 'Intersection',
    'UNKNOWN INTERSECTION TYPE': 'Intersection',

    'FOUR WAY': 'Four-Way',
    'ONE-WAY': 'One-Way',

    'PARKING LOT': 'Parking Lot',
    'DRIVEWAY': 'Driveway',

    'TRAFFIC ROUTE': 'Traffic Route',
    'ALLEY': 'Alley',
    'Ramp': 'Ramp',

    'FIVE POINT, OR MORE': 'Turn Lane',
    'CENTER TURN LANE': 'Turn Lane',
    'ROUNDABOUT': 'Roundabout',

    'NOT REPORTED': 'Unknown',
    'UNKNOWN': 'Unknown',
    'OTHER': 'Unknown',
})

In [83]:
# collision, fixed, movement, nonmotorist
df_cp4['FIRST_CRASH_TYPE'] = df_cp4['FIRST_CRASH_TYPE'].replace(
    {
        'SIDESWIPE SAME DIRECTION': 'Collision',
        'SIDESWIPE OPPOSITE DIRECTION': 'Collision',
        'REAR END': 'Collision',
        'REAR TO SIDE': 'Collision',
        'HEAD ON': 'Collision',
        'REAR TO FRONT': 'Collision',
        'OTHER OBJECT': 'Collision',
        'REAR TO REAR': 'Collision',

        'TURNING': 'Collision',
        'ANGLE': 'Collision',
        'OVERTURNED': 'Collision',

        'PARKED MOTOR VEHICLE': 'Fixed',
        'FIXED OBJECT': 'Fixed',
        'TRAIN': 'Fixed',

        'PEDALCYCLIST': 'Pedestrian',
        'PEDESTRIAN': 'Pedestrian',
        'ANIMAL': 'Pedestrian',

        'OTHER NONCOLLISION': 'Other',
    }
)
df_cp4 = df_cp4.rename({'FIRST_CRASH_TYPE': 'CRASH_TYPE'}, axis=1)
df_cp4.loc[:, 'CRASH_TYPE']

0           Collision
1           Collision
2               Fixed
3          Pedestrian
4           Collision
              ...    
1010921         Fixed
1010922     Collision
1010923     Collision
1010924     Collision
1010925     Collision
Name: CRASH_TYPE, Length: 1010926, dtype: object

In [84]:
df_cp4['ROAD_DEFECT'].unique()

array(['NO DEFECTS', 'UNKNOWN', 'SHOULDER DEFECT', 'WORN SURFACE',
       'RUT, HOLES', 'OTHER', 'DEBRIS ON ROADWAY'], dtype=object)

In [85]:
df_cp4['ROAD_DEFECT'] = df_cp4['ROAD_DEFECT'].replace(
    {
        'NO DEFECTS': False,
        'UNKNOWN': True,
        'SHOULDER DEFECT': True,
        'WORN SURFACE': True,
        'RUT, HOLES': True,
        'OTHER': True,
        'DEBRIS ON ROADWAY': True
    }
)

/var/folders/_9/nyjys43x4nv6dxyk3y75vzh80000gn/T/ipykernel_38598/2626659515.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_cp4['ROAD_DEFECT'] = df_cp4['ROAD_DEFECT'].replace(


In [86]:
df_cp4['ROADWAY_SURFACE_COND'] = df_cp4['ROADWAY_SURFACE_COND'].replace(
    {
        'SNOW OR SLUSH': 'Snow',
        'DRY': 'Dry',
        'WET': 'Wet',
        'ICE': 'Ice',
        'SAND, MUD, DIRT': 'Dirt/Mud',
        'OTHER': 'Unknown',
        'UNKNOWN': 'Unknown',
    }
)

In [87]:
df_cp4['PRIM_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER TURNING/NO SIGNAL', 'IMPROPER OVERTAKING/PASSING',
       'UNABLE TO DETERMINE', 'FOLLOWING TOO CLOSELY',
       'DISREGARDING TRAFFIC SIGNALS', 'IMPROPER LANE USAGE',
       'FAILING TO YIELD RIGHT-OF-WAY', 'NOT APPLICABLE',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'DISREGARDING STOP SIGN', 'IMPROPER BACKING',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'TURNING RIGHT ON RED', 'PHYSICAL CONDITION OF DRIVER',
       'EQUIPMENT - VEHICLE CONDITION', 'DRIVING ON WRONG SIDE/WRONG WAY',
       'DISREGARDING ROAD MARKINGS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'DISTRACTION - OTHER ELECTRON

In [88]:
contributory_cause_remap = {
    'IMPROPER TURNING/NO SIGNAL': 'Improper Turning',
    'IMPROPER OVERTAKING/PASSING': 'Improper Overtaking',
    'IMPROPER LANE USAGE': 'Improper Lane Usage',
    'IMPROPER BACKING': 'Improper Backing',

    'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE': 'Inexperience',
    'DRIVING ON WRONG SIDE/WRONG WAY': 'Driving Wrong Way',

    'FOLLOWING TOO CLOSELY': 'Tailgating',

    'DISREGARDING STOP SIGN': 'Ignoring Traffic Sign',
    'DISREGARDING TRAFFIC SIGNALS': 'Ignoring Traffic Sign',
    'DISREGARDING ROAD MARKINGS': 'Ignoring Traffic Sign',
    'DISREGARDING OTHER TRAFFIC SIGNS': 'Ignoring Traffic Sign',
    'DISREGARDING YIELD SIGN': 'Ignoring Traffic Sign',

    'FAILING TO YIELD RIGHT-OF-WAY': 'Failing to Yield Right-of-Way',
    'TURNING RIGHT ON RED': 'Turning on Red',

    'PHYSICAL CONDITION OF DRIVER': 'Driver Physical Condition',

    'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)': 'Driving Under the Influence (Arrest)',
    'HAD BEEN DRINKING (USE WHEN ARREST IS NOT MADE)': 'Driving Under the Influence (No Arrest)',

    'WEATHER': 'Weather',
    'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER': 'Aggressive Driving',

    'EQUIPMENT - VEHICLE CONDITION': 'Vehicle Condition',


    'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)': 'Vision Obscured',
    'OBSTRUCTED CROSSWALKS': 'Obstacle',
    'ROAD ENGINEERING/SURFACE/MARKING DEFECTS': 'Road Defects',

    'DISTRACTION - FROM OUTSIDE VEHICLE': 'Distracted Driver',
    'DISTRACTION - FROM INSIDE VEHICLE': 'Distracted Driver',
    'DISTRACTION - OTHER ELECTRONIC DEVICE (NAVIGATION DEVICE, DVD PLAYER, ETC.)': 'Distracted Driver',
    'CELL PHONE USE OTHER THAN TEXTING': 'Distracted Driver',
    'TEXTING': 'Driver Texting',

    'ROAD CONSTRUCTION/MAINTENANCE': 'Road Construction',

    'RELATED TO BUS STOP': 'Related to Bus Stop',
    'PASSING STOPPED SCHOOL BUS': 'Passing Stopped School Bus',

    'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST': 'Evasive Maneuver',
    'ANIMAL': 'Animal',

    'BICYCLE ADVANCING LEGALLY ON RED LIGHT': '2-Wheel Legally Advancing',
    'MOTORCYCLE ADVANCING LEGALLY ON RED LIGHT': '2-Wheel Legally Advancing',

    'FAILING TO REDUCE SPEED TO AVOID CRASH': 'Speeding',
    'EXCEEDING AUTHORIZED SPEED LIMIT': 'Speeding',
    'EXCEEDING SAFE SPEED FOR CONDITIONS': 'Speeding',

    'NOT APPLICABLE': 'Unknown',
    'UNABLE TO DETERMINE': 'Unknown',
 }

In [89]:
df_cp4['SEC_CONTRIBUTORY_CAUSE'].unique()

array(['IMPROPER OVERTAKING/PASSING', 'UNABLE TO DETERMINE',
       'FOLLOWING TOO CLOSELY', 'NOT APPLICABLE',
       'FAILING TO YIELD RIGHT-OF-WAY',
       'DRIVING SKILLS/KNOWLEDGE/EXPERIENCE', 'WEATHER',
       'FAILING TO REDUCE SPEED TO AVOID CRASH',
       'PHYSICAL CONDITION OF DRIVER', 'IMPROPER LANE USAGE',
       'OPERATING VEHICLE IN ERRATIC, RECKLESS, CARELESS, NEGLIGENT OR AGGRESSIVE MANNER',
       'IMPROPER BACKING', 'DISREGARDING STOP SIGN',
       'UNDER THE INFLUENCE OF ALCOHOL/DRUGS (USE WHEN ARREST IS EFFECTED)',
       'IMPROPER TURNING/NO SIGNAL', 'ROAD CONSTRUCTION/MAINTENANCE',
       'DRIVING ON WRONG SIDE/WRONG WAY', 'DISREGARDING TRAFFIC SIGNALS',
       'EVASIVE ACTION DUE TO ANIMAL, OBJECT, NONMOTORIST',
       'EQUIPMENT - VEHICLE CONDITION',
       'CELL PHONE USE OTHER THAN TEXTING',
       'VISION OBSCURED (SIGNS, TREE LIMBS, BUILDINGS, ETC.)',
       'DISTRACTION - FROM OUTSIDE VEHICLE',
       'ROAD ENGINEERING/SURFACE/MARKING DEFECTS', 'TEXTING',
  

In [90]:
df_cp4['PRIM_CONTRIBUTORY_CAUSE'] = df_cp4['PRIM_CONTRIBUTORY_CAUSE'].replace(contributory_cause_remap)
df_cp4['SEC_CONTRIBUTORY_CAUSE'] = df_cp4['SEC_CONTRIBUTORY_CAUSE'].replace(contributory_cause_remap)

In [91]:
x = pd.Series(df_cp4['PRIM_CONTRIBUTORY_CAUSE'].unique() + df_cp4['SEC_CONTRIBUTORY_CAUSE'].unique())
x.unique()

array(['Improper TurningImproper Overtaking',
       'Improper OvertakingUnknown', 'UnknownTailgating',
       'TailgatingFailing to Yield Right-of-Way',
       'Ignoring Traffic SignInexperience', 'Improper Lane UsageWeather',
       'Failing to Yield Right-of-WaySpeeding',
       'Driving Under the Influence (Arrest)Driver Physical Condition',
       'SpeedingImproper Lane Usage', 'InexperienceAggressive Driving',
       'WeatherImproper Backing', 'Improper BackingIgnoring Traffic Sign',
       'Aggressive DrivingDriving Under the Influence (Arrest)',
       'Turning on RedImproper Turning',
       'Driver Physical ConditionRoad Construction',
       'Vehicle ConditionDriving Wrong Way',
       'Driving Wrong WayEvasive Maneuver',
       'Evasive ManeuverVehicle Condition',
       'Road DefectsDistracted Driver', 'Vision ObscuredVision Obscured',
       'Distracted DriverRoad Defects', 'Driver TextingDriver Texting',
       'Road ConstructionTurning on Red',
       'Driving Under the

In [92]:
df_cp4['LIGHTING_CONDITION'].unique()

array(['DAYLIGHT', 'UNKNOWN', 'DARKNESS, LIGHTED ROAD', 'DARKNESS',
       'DUSK', 'DAWN'], dtype=object)

In [93]:
df_cp4['LIGHTING_CONDITION'] = df_cp4['LIGHTING_CONDITION'].replace(
    {
        'DAYLIGHT': "Day",
        'DAWN': "Day",
        'DARKNESS, LIGHTED ROAD': "Night",
        'DARKNESS': "Night",
        'DUSK': "Night",
        'UNKNOWN': "NA"
    }
)

## Renaming a few more columns

In [94]:
df_cp4 = df_cp4.rename({'ALIGNMENT': 'ROAD_LEVEL', 'DAMAGE': 'DAMAGE_AMT', 'ROADWAY_SURFACE_COND': 'ROAD_CONDITION'}, axis=1)

## Checking street columns

In [95]:
# 'STREET_NAME',
# 'STREET_NO',
# 'STREET_DIRECTION',

street_names = []

for idx, item in df_cp4['STREET_NAME'].items():
    street_names.append(f'{df_cp4.loc[idx, "STREET_NO"]} {df_cp4.loc[idx, "STREET_NAME"]}')

df_cp4.loc[:, 'STREET_NAME'] = street_names
df_cp4['STREET_NAME'] = df_cp4['STREET_NAME'].str.title()

In [96]:
df_cp4['STREET_NAME']

0            6352 Sheridan Rd
1            3555 Western Ave
2              1005 Drake Ave
3            2901 Division St
4               4001 111Th St
                  ...        
1010921     898 Lake Shore Dr
1010922        2416 Damen Ave
1010923         3700 Broadway
1010924        1800 Union Ave
1010925    2357 Milwaukee Ave
Name: STREET_NAME, Length: 1010926, dtype: object

In [97]:
df_cp4 = df_cp4.drop('STREET_NO', axis=1)

## Ensuring correct datatypes

In [98]:
df_cp4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1010926 entries, 0 to 1010925
Data columns (total 30 columns):
 #   Column                         Non-Null Count    Dtype         
---  ------                         --------------    -----         
 0   CRASH_RECORD_ID                1010926 non-null  object        
 1   CRASH_DATE                     1010926 non-null  datetime64[ns]
 2   POSTED_SPEED_LIMIT             1010926 non-null  int64         
 3   TRAFFIC_CONTROL_DEVICE         1010926 non-null  object        
 4   TRAFFIC_DEVICE_FUNCTIONING     1010926 non-null  bool          
 5   WEATHER_CONDITION              1010926 non-null  object        
 6   LIGHTING_CONDITION             1010926 non-null  object        
 7   CRASH_TYPE                     1010926 non-null  object        
 8   TRAFFICWAY_TYPE                1010926 non-null  object        
 9   ROAD_LEVEL                     1010926 non-null  object        
 10  ROAD_CONDITION                 1010926 non-null  objec

In [99]:
types = {
    'CRASH_DATE': None,
     'POSTED_SPEED_LIMIT': int,
     'TRAFFIC_CONTROL_DEVICE': None,
     'TRAFFIC_DEVICE_FUNCTIONING': bool,
     'WEATHER_CONDITION': None,
     'LIGHTING_CONDITION': None,
     'CRASH_TYPE': None,
     'TRAFFICWAY_TYPE': None,
     'ROAD_LEVEL': None,
     'ROAD_CONDITION': None,
     'ROAD_DEFECT': bool,
     'ON_SCENE': int,
     'CRASH_SEVERE': bool,
     'HIT_AND_RUN': bool,
     'DAMAGE_AMT': None,
     'DATE_POLICE_NOTIFIED': None,
     'PRIM_CONTRIBUTORY_CAUSE': None,
     'SEC_CONTRIBUTORY_CAUSE': None,
     'INJURIES_TOTAL': None,
     'INJURIES_FATAL': None,
     'INJURIES_INCAPACITATING': None,
     'INJURIES_NON_INCAPACITATING': None,
     'INJURIES_REPORTED_NOT_EVIDENT': None,
     'INJURIES_NO_INDICATION': None,
     'INJURIES_UNKNOWN': int,
     'LATITUDE': None,
     'LONGITUDE': None,
}

for col, typ in types.items():
    if typ is not None:
        try:
            df_cp4[col] = df_cp4[col].astype(dtype=typ)
        except:
            log.error(col)

Prettifying the text

In [100]:
df_cp4['ROAD_CONDITION'] = df_cp4['ROAD_CONDITION'].str.title()
df_cp4['ROAD_LEVEL'] = df_cp4['ROAD_LEVEL'].str.title()

# Final Cleaned Dataset

In [101]:
df_CLEANED = df_cp4.copy()

## Reorder columns

In [102]:
# reordered_columns = [
#     'CRASH_DATE',
#     'DATE_POLICE_NOTIFIED',
#
#     'ON_SCENE',
#     'HIT_AND_RUN',
#
#     'POSTED_SPEED_LIMIT',
#
#     'TRAFFIC_CONTROL_DEVICE',
#     'TRAFFIC_DEVICE_FUNCTIONING',
#     'TRAFFICWAY_TYPE',
#
#     'WEATHER_CONDITION',
#     'LIGHTING_CONDITION',
#     'ROAD_DEFECT',
#     'ROAD_LEVEL',
#     'ROAD_CONDITION',
#
#     'CRASH_TYPE',
#     'CRASH_SEVERE',
#
#     'PRIM_CONTRIBUTORY_CAUSE',
#     'SEC_CONTRIBUTORY_CAUSE',
#
#     'INJURIES_TOTAL',
#     'INJURIES_FATAL',
#     'INJURIES_INCAPACITATING',
#     'INJURIES_NON_INCAPACITATING',
#     'INJURIES_REPORTED_NOT_EVIDENT',
#     'INJURIES_NO_INDICATION',
#     'INJURIES_UNKNOWN',
#
#     'DAMAGE_AMT',
#
#     'STREET_NAME',
#     'STREET_DIRECTION',
#
#     'LATITUDE',
#     'LONGITUDE',
# ]
# df_cpdf_CLEANED5 = df_CLEANED[reordered_columns]
reordered_columns = df_CLEANED.columns.tolist()
reordered_columns.append(reordered_columns.pop(0))
reordered_columns

['CRASH_DATE',
 'POSTED_SPEED_LIMIT',
 'TRAFFIC_CONTROL_DEVICE',
 'TRAFFIC_DEVICE_FUNCTIONING',
 'WEATHER_CONDITION',
 'LIGHTING_CONDITION',
 'CRASH_TYPE',
 'TRAFFICWAY_TYPE',
 'ROAD_LEVEL',
 'ROAD_CONDITION',
 'ROAD_DEFECT',
 'ON_SCENE',
 'CRASH_SEVERE',
 'HIT_AND_RUN',
 'DAMAGE_AMT',
 'DATE_POLICE_NOTIFIED',
 'PRIM_CONTRIBUTORY_CAUSE',
 'SEC_CONTRIBUTORY_CAUSE',
 'STREET_DIRECTION',
 'STREET_NAME',
 'INJURIES_TOTAL',
 'INJURIES_FATAL',
 'INJURIES_INCAPACITATING',
 'INJURIES_NON_INCAPACITATING',
 'INJURIES_REPORTED_NOT_EVIDENT',
 'INJURIES_NO_INDICATION',
 'INJURIES_UNKNOWN',
 'LATITUDE',
 'LONGITUDE',
 'CRASH_RECORD_ID']

In [103]:
df_CLEANED = df_CLEANED[reordered_columns]
df_CLEANED

,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,TRAFFIC_DEVICE_FUNCTIONING,WEATHER_CONDITION,LIGHTING_CONDITION,CRASH_TYPE,TRAFFICWAY_TYPE,ROAD_LEVEL,ROAD_CONDITION,...,INJURIES_TOTAL,INJURIES_FATAL,INJURIES_INCAPACITATING,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,LATITUDE,LONGITUDE,CRASH_RECORD_ID
0,2025-01-14 12:25:00,30,Traffic Signal,True,Snow,Day,Collision,Divided,Straight And Level,Snow,...,0,0,0,0,0,2,0,41.997808,-87.655770,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...
1,2025-05-23 09:30:00,30,Stop Sign,False,Unknown,Day,Collision,Divided,Straight And Level,Unknown,...,0,0,0,0,0,2,0,41.946529,-87.688106,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...
2,2025-04-05 20:00:00,30,No Controls,False,Clear,NA,Fixed,Not Divided,Straight And Level,Dry,...,0,0,0,0,0,1,0,41.899325,-87.715074,04d91dffc94f677358ca47056921ba5c4224320df27ed4...
3,2025-05-23 09:15:00,30,No Controls,False,Clear,Day,Pedestrian,Not Divided,Straight And Level,Dry,...,1,0,0,1,0,2,0,41.902793,-87.699412,0b5603954d84b7341c7cad4f570ea039e85919f3750ccb...
4,2025-01-14 08:00:00,30,Traffic Signal,True,Snow,Day,Collision,Four-Way,Straight And Level,Wet,...,0,0,0,0,0,2,0,41.691207,-87.720555,00bce77960c2faa2a8782a8cac1d6e5715802d6c072a08...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010921,2025-10-18 03:50:00,35,No Controls,False,Rain,Night,Fixed,Not Divided,Straight And Level,Wet,...,0,0,0,0,0,1,0,41.899200,-87.618841,7a7963ac15a38f3cf7549397dd053fd7612dd5194de280...
1010922,2025-10-18 17:08:00,30,No Controls,False,Clear,Day,Collision,Not Divided,Straight And Level,Dry,...,0,0,0,0,0,5,0,41.848081,-87.675881,84e546c12fd8fd78ae4e2dc26fdb02d6e8f6c4c92be450...
1010923,2025-10-19 01:50:00,30,Unknown,False,Unknown,NA,Collision,Not Divided,Straight And Level,Unknown,...,0,0,0,0,0,5,0,41.949157,-87.648527,8237045757e7fed5adfb450bd7c8472e28a61d322f86bb...
1010924,2023-07-10 12:29:00,30,Traffic Signal,True,Clear,Day,Collision,Four-Way,Straight And Level,Dry,...,0,0,0,0,0,2,0,41.857531,-87.644929,61c8dcd63fae60613bc9ec526fa901420cbe99a6d35840...


## Export to CSV

In [104]:
lib.export_data(filename, df_CLEANED)

PortfolioLogger.lib.tools: INFO: Exporting 30327780 elements (940.65 MB) to chicago_traffic_crashes_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10afcd9e0> took 6.087 secs to complete.


PosixPath('chicago_traffic_crashes_CLEAN.csv')